# 03 — Fisher block-rank diagnostic: powder rank-deficient → joint full-rank

This is the **headline result of the paper**.  For the per-panel
`(δy, δz)` block:

* **Powder-only** data Fisher is **rank-deficient** — a single
  calibrant image cannot independently determine every panel's shift
  (paper-3 §9 result; some panels under-/un-sampled by rings).
* **HEDM evidence** adds independent constraints (spots from many
  grains hit panels from many directions).
* **Joint** powder + HEDM is **full-rank** on the covered panels.

We compute `fisher_block_rank` on the gauge-free **data** Fisher under
each modality.  Self-contained synthetic; runs in seconds (no LM).


In [1]:
import os, math, time
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')   # macOS OpenMP guard
import numpy as np
import torch

import midas_peakfit as mp
from midas_calibrate_v2.forward.panels import PanelLayout
from midas_hkls import Lattice, SpaceGroup
from midas_diffract.hkls import hkls_for_forward_model

# The package's validated synthetic generators (paper fig. 1 source).
import midas_joint_ff_calibrate.runners.run_synthetic_pilatus_joint as R
from midas_joint_ff_calibrate.loss import JointWeights, joint_residual
from midas_joint_ff_calibrate.pipelines.identifiability import fisher_block_rank
from midas_joint_ff_calibrate.pipelines.alternating import AlternatingDriver
from midas_joint_ff_calibrate.pipelines.full_joint import FullJointDriver

# ---- shrink the problem for notebook speed (production paths unchanged)
R.N_GRAINS = 24
R.N_PANELS_Y = 4
R.N_PANELS_Z = 4
R.PANEL_SIZE_Y = 150
R.PANEL_SIZE_Z = 150
R.LSD_UM = 7.0e5            # closer detector -> more panels see Au rings
R.N_RINGS = 6
R.TWO_THETA_MAX_DEG = 14.0
R.N_POWDER_PER_RING = 180

# Loss weights (1/sigma per modality so neither dominates) + gauge lambda.
W_POWDER, W_HEDM, LAMBDA_GAUGE = 1.0e4, 10.0, 1.0e6


def build_problem(seed: int = 2026):
    """Build (layout, truth, grains, ring 2theta/d, powder+HEDM obs)."""
    layout = PanelLayout.regular(R.N_PANELS_Y, R.N_PANELS_Z,
                                 R.PANEL_SIZE_Y, R.PANEL_SIZE_Z,
                                 gap_y=R.GAP_Y, gap_z=R.GAP_Z)
    truth = R.sample_truth(layout, seed=seed)
    grain_eulers, grain_pos, grain_lat = R.sample_truth_grains(seed=seed + 1)

    sg = SpaceGroup.from_number(225)                 # Fm-3m (Au)
    lat = Lattice.for_system('cubic', a=R.AU_LATTICE_A)
    _, thetas, _ = hkls_for_forward_model(
        sg, lat, wavelength_A=R.WAVELENGTH_A,
        two_theta_max_deg=R.TWO_THETA_MAX_DEG, expand_equivalents=False)
    ring_tt, _ = torch.unique(2 * thetas * 180.0 / math.pi,
                              return_inverse=True, sorted=True)
    ring_tt = ring_tt.double()[:R.N_RINGS]
    ring_d = R.WAVELENGTH_A / (2.0 * torch.sin(ring_tt * math.pi / 360.0))

    powder_obs = R.generate_powder_observations(layout, truth, ring_tt, seed=seed + 2)
    hedm_obs = R.generate_hedm_observations(
        layout, truth, grain_eulers, grain_pos, grain_lat, seed=seed + 3)
    return dict(layout=layout, truth=truth,
                grain_eulers=grain_eulers, grain_pos=grain_pos, grain_lat=grain_lat,
                ring_tt=ring_tt, ring_d=ring_d,
                powder_obs=powder_obs, hedm_obs=hedm_obs)


def build_spec(prob, *, refine_grains=False, refine_panels=True):
    """Canonical joint spec: geometry + per-panel deltas + grain blocks."""
    truth = prob['truth']; layout = prob['layout']
    spec = mp.ParameterSpec()
    spec.add(mp.Parameter('Lsd', init=truth.Lsd + 50.0,
                          bounds=(truth.Lsd - 5e3, truth.Lsd + 5e3)))
    spec.add(mp.Parameter('BC_y', init=truth.BC_y + 0.3,
                          bounds=(truth.BC_y - 5.0, truth.BC_y + 5.0)))
    spec.add(mp.Parameter('BC_z', init=truth.BC_z - 0.2,
                          bounds=(truth.BC_z - 5.0, truth.BC_z + 5.0)))
    spec.add(mp.Parameter('ty', init=0.0, refined=False))
    spec.add(mp.Parameter('tz', init=0.0, refined=False))
    spec.add(mp.Parameter('Wavelength', init=R.WAVELENGTH_A, refined=False))
    spec.add(mp.Parameter('pxY', init=R.PX_UM, refined=False))
    spec.add(mp.Parameter('pxZ', init=R.PX_UM, refined=False))
    spec.add(mp.Parameter('RhoD', init=200000.0, refined=False))
    spec.add(mp.Parameter('panel_delta_yz',
                          init=torch.zeros(layout.n_panels(), 2, dtype=torch.float64),
                          bounds=(-3.0, 3.0),
                          prior=mp.GaussianPrior(mean=0.0, std=0.5),
                          refined=refine_panels))
    spec.add(mp.Parameter('panel_delta_theta',
                          init=torch.zeros(layout.n_panels(), dtype=torch.float64),
                          refined=False))
    spec.add(mp.Parameter('grain_euler', init=prob['grain_eulers'],
                          bounds=(-2 * math.pi, 2 * math.pi), refined=refine_grains))
    spec.add(mp.Parameter('grain_pos', init=prob['grain_pos'],
                          bounds=(-1000.0, 1000.0), refined=refine_grains))
    spec.add(mp.Parameter('grain_lattice', init=prob['grain_lat'], refined=False))
    return spec


def make_closures(prob, spec):
    """Return (joint, powder_only, hedm_only) gauge-free residual closures."""
    pf = R.make_powder_residual(prob['powder_obs'], prob['layout'],
                                prob['ring_tt'], ring_d_spacing_A=prob['ring_d'])
    hf = R.make_hedm_residual(prob['hedm_obs'], prob['layout'])

    def joint_fn(u):
        return joint_residual(
            u, powder_residual_fn=pf, hedm_residual_fn=hf, spec=spec,
            weights=JointWeights(w_powder=W_POWDER, w_hedm=W_HEDM,
                                 lambda_gauge=LAMBDA_GAUGE),
            gauge_blocks=[])

    def powder_only(u):   return W_POWDER * pf(u)
    def hedm_only(u):     return W_HEDM * hf(u)
    return joint_fn, powder_only, hedm_only


def seed_unpacked(spec):
    """Dict of every parameter at its init value, as float64 tensors."""
    u = {n: spec.parameters[n].init_tensor() for n in spec.parameters}
    for n in u:
        if not isinstance(u[n], torch.Tensor):
            u[n] = torch.tensor(u[n], dtype=torch.float64)
    return u


def covered_panels(prob):
    p = set(prob['powder_obs']['panel_idx'].tolist())
    h = set(prob['hedm_obs']['panel_idx'].tolist())
    return p, h, (p | h)


DIPlib -- a quantitative image analysis library
Version 3.5.2 (Dec 27 2024)
For more information see https://diplib.org


/Users/hsharma/miniconda3/envs/midas_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Build a prior-free spec for the data-only rank diagnostic

The rank-deficiency is a property of the **data alone**.  A Gaussian
prior on `panel_delta_yz` (used in the production fit, notebook 02)
adds curvature in *every* panel direction and would mask the
deficiency — so for this diagnostic we build the panel block with **no
prior** (and plain bounds), exposing the raw data information content.


In [2]:
prob = build_problem()
layout = prob['layout']

# Prior-free spec: identical to build_spec but panel_delta_yz has no prior.
spec = build_spec(prob, refine_grains=False, refine_panels=True)
del spec.parameters['panel_delta_yz']
spec.add(mp.Parameter('panel_delta_yz',
                      init=torch.zeros(layout.n_panels(), 2, dtype=torch.float64),
                      bounds=(-3.0, 3.0)))             # NO prior
joint_fn, powder_only, hedm_only = make_closures(prob, spec)

p_pan, h_pan, cov = covered_panels(prob)
n_block = 2 * layout.n_panels()           # (dy, dz) per panel
n_covered = len(cov)
print(f'{layout.n_panels()} panels; powder sees {len(p_pan)}, '
      f'HEDM sees {len(h_pan)}, union {n_covered}')
print(f'panel_delta_yz block size = {n_block} '
      f'(full rank on covered = 2*{n_covered} = {2*n_covered})')


16 panels; powder sees 12, HEDM sees 12, union 12
panel_delta_yz block size = 32 (full rank on covered = 2*12 = 24)


## Data Fisher rank under each modality

A full-rank covered block has rank `2 * n_covered`.  Powder-only falls
short; HEDM and joint reach it.


In [3]:
seed_unp = seed_unpacked(spec)
reports = {}
print(f'{"modality":<12s}  {"rank":>6s}  {"of":>5s}  {"cond":>11s}  '
      f'{"sigma_med":>11s}')
for label, fn in [('powder-only', powder_only),
                  ('hedm-only', hedm_only),
                  ('joint', joint_fn)]:
    rep = fisher_block_rank(spec, fn, seed_unp,
                            block_names=['panel_delta_yz'],
                            sigma_r=1.0, fallback_span=2.0)
    reports[label] = rep
    sig = rep.sigma_per_dim
    print(f'{label:<12s}  {rep.rank:>6d}  {sig.numel():>5d}  '
          f'{rep.condition_number:>11.2e}  {float(sig.median()):>11.3e}')

print(f'\n2*covered = {2*n_covered}')
print(f"powder-only rank-deficient vs covered: "
      f"{reports['powder-only'].rank < 2*n_covered}")
print(f"joint reaches full rank on covered:    "
      f"{reports['joint'].rank == 2*n_covered}")


modality        rank     of         cond    sigma_med


powder-only       22     32    5.90e+304    4.384e-02
hedm-only         24     32    1.24e+304    2.556e-02
joint             24     32    7.13e+304    3.396e-02

2*covered = 24
powder-only rank-deficient vs covered: True
joint reaches full rank on covered:    True


## Why the difference?

* The **powder** ring at radius `R` only constrains the panel shift
  along the radial direction at the η where the ring crosses that
  panel.  Panels with no ring crossing (or only a tangential one) are
  rank-deficient.
* **HEDM spots** from many grains strike each panel from many
  directions, constraining both `δy` and `δz` per panel.
* The **joint** loss stacks both residuals, so the combined Jacobian
  spans the full per-panel space wherever *either* modality sees a
  panel.

## The nullspace directions

`fisher_block_rank` returns the rank-deficient directions for the
powder-only block — the panel-shift combinations the powder data
cannot see.


In [4]:
null = reports['powder-only'].nullspace_directions
print(f'powder-only nullspace: {null.shape[0]} unconstrained direction(s) '
      f'in the {null.shape[1]}-dim panel block')
if null.shape[0] > 0:
    # Which panels dominate the first nullspace direction?
    d0 = null[0].reshape(layout.n_panels(), 2).norm(dim=1)
    top = torch.argsort(d0, descending=True)[:5]
    print('panels with largest weight in nullspace dir 0:',
          [int(t) for t in top])


powder-only nullspace: 10 unconstrained direction(s) in the 32-dim panel block
panels with largest weight in nullspace dir 0: [0, 1, 2, 3, 4]


## Takeaway

The figure the paper leads with: **powder-only is rank-deficient on
the per-panel block; HEDM evidence makes the joint problem full-rank.**
This is *why* joint calibration is necessary for densely-tiled
multi-panel detectors (Pilatus / Eiger).
